# Compile results (reduced) — results_auto.xlsx + real-life ssd `_grid.pdf` charts

Reduced version of `compile_results.ipynb`, scoped to exactly the two outputs
this repo's paper actually uses: `results_auto.xlsx` and the real-life (ssd)
best-of-category `_grid.pdf` charts (`real_ssd_bestcat_cc_grid.pdf` /
`real_ssd_bestcat_tt_grid.pdf`).

Dropped vs. the full `compile_results.ipynb`: the "COMPILE ALL RESULTS"
full-corpus amiri XES scan (slow, and not a dependency of either kept
output), `export_viz_excel`/`export_viz_pdf`, every non-grid bar chart /
boxplot / NMAE variant, and the 3 extra plot functions
(`plot_relative_mae_heatmap`/`plot_family_dotplot_grid`/
`plot_paired_cc_tt_grid`) not wired into either kept output.


In [ ]:
from pathlib import Path
import sys, warnings
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

ROOT        = Path('.').resolve().parent.parent  # pipelines/compile_results/ -> repo root
sys.path.insert(0, str(ROOT))
BEST_MODELS = ROOT / 'best_models'

from setttings import set_global_seed
set_global_seed(1904)


## `results_auto.xlsx`

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# EXPORT RESULTS TO EXCEL
# ═══════════════════════════════════════════════════════════════════════════════
# Builds results_auto.xlsx matching the structure of results_overall.xlsm.
#
# Sheets  : none, peak_0.6, peak_0.7, peak_0.8, magnitude_1, magnitude_2, magnitude_3
# 'none'  : Synthetic (Loan + O2C) on the left, Real-Life on the right
# Others  : Real-Life only
# Layout  : Concurrent Cases block on top, Avg Throughput Time block on bottom
# Columns : one mse+rmse+mae triple per dataset; Best Value count at the right end
# Highlight: green fill on column-wise minimum (best model per dataset/metric)
# ═══════════════════════════════════════════════════════════════════════════════

def export_results_excel(out_path=None):
    import numpy as np
    try:
        from openpyxl import Workbook
        from openpyxl.styles import Font, PatternFill, Alignment
        from openpyxl.utils import get_column_letter
    except ImportError:
        raise ImportError("pip install openpyxl")

    RESULTS  = ROOT / 'results'
    out_path = Path(out_path) if out_path else RESULTS / 'results_auto.xlsx'

    # ── Dataset / trim / model definitions ──────────────────────────────────
    SYNTH_LOAN  = ['loan_flat', 'loan_seasonal', 'loan_trend', 'loan_drift', 'loan_combined']
    SYNTH_O2C   = ['o2c_flat',  'o2c_seasonal',  'o2c_trend',  'o2c_drift',  'o2c_combined']
    SHORT_NAMES = ['flat', 'seasonal', 'trend', 'drift', 'combined']
    REAL_LIFE   = ['bpic12-a', 'bpic15-1', 'bpic15-2', 'bpic17-o',
                   'bpic20-dom', 'bpic20-int', 'helpdesk', 'sepsis']
    TRIM_SHEETS = ['none', 'peak_0.6', 'peak_0.7', 'peak_0.8',
                   'magnitude_1', 'magnitude_2', 'magnitude_3', 'ssd']
    SERIES_LIST = [('Concurrent Cases',    'concurrent_cases'),
                   ('Avg Throughput Time', 'throughput_time')]

    # (display_name, source_type, detail)
    #   'baseline' → model name in metrics_{ds}.csv
    #   'val_mean' → looked up in val_mean_all.csv
    #   'pmsd'     → looked up in metrics_{ds}_pmsd.csv (never merged into the
    #                main metrics_{ds}.csv — same convention as val_mean)
    #   'process'  → (results_subdir, metrics_file_suffix)
    MODELS = [
        # ── Time series baselines ────────────────────────────────────────────
        ('naive',                 'baseline', 'naive'),
        ('seasonal_naive',        'baseline', 'seasonal_naive'),
        ('val_average',           'val_mean',  None),
        ('ets',                   'baseline', 'ets'),
        ('sarimax',               'baseline', 'sarimax'),
        ('theta',                 'baseline', 'theta'),
        ('stl',                   'baseline', 'stl'),
        ('ridge',                 'baseline', 'ridge'),
        ('ridge_mimo',            'baseline', 'ridge_mimo'),
        ('gru',                   'baseline', 'gru'),
        ('gru_mimo',              'baseline', 'gru_mimo'),
        ('nbeats',                'baseline', 'nbeats'),
        ('nhits',                 'baseline', 'nhits'),
        ('tft',                   'baseline', 'tft'),
        ('prophet',               'baseline', 'prophet'),
        # ── Tabular foundation models ────────────────────────────────────────
        ('chronos',               'foundation', 'chronos'),
        ('tabpfn',                'foundation', 'tabpfn'),
        # ── Process simulation ───────────────────────────────────────────────
        ('PMSD',                  'pmsd',       'pmsd'),
        ('Simod',                 'simulator',  'simod'),
        ('AgentSimulator',        'simulator',  'agentsimulator'),
        # ── Process models — plain-field (zero-knowledge) ────────────────────
        ('PF Amiri',              'process',  ('plain_field/amiri',        '')),
        ('PF Camargo',            'process',  ('plain_field/camargo',      '')),
        ('PF Bukhsh RT',          'process',  ('plain_field/bukhsh_rt', '', 'plain_field/bukhsh')),
        ('PF Bukhsh suffix',      'process',  ('plain_field/bukhsh_suffix', '')),
        # ── Process models — full trace ──────────────────────────────────────
        ('GLSTM full',            'process',  ('camargo_hpo',      '')),
        ('Amiri full',            'process',  ('amiri_hpo',        '')),
        ('PT suffix full',        'process',  ('bukhsh_hpo',       '')),
        ('PT RT full',            'process',  ('bukhsh_hpo',       '_rt')),
        # ── Process models — half prefix ─────────────────────────────────────
        ('GLSTM half prefix',     'process',  ('camargo_hpo_half', '')),
        ('Amiri half',            'process',  ('amiri_hpo_half',   '')),
        ('PT suffix half prefix', 'process',  ('bukhsh_hpo_half',  '')),
        ('PT RT half prefix',     'process',  ('bukhsh_hpo_half',  '_rt')),
    ]
    N_MODELS = len(MODELS)

    # ── Load val_mean reference file ─────────────────────────────────────────
    _vm_df = pd.DataFrame()
    _vm_p  = RESULTS / 'val_mean_all.csv'
    if _vm_p.exists():
        _vm_df = pd.read_csv(_vm_p)

    # ── Cached loaders ───────────────────────────────────────────────────────
    _bl_cache = {}
    _pr_cache = {}
    _sim_cache = {}

    def _find_baseline_csv(directory, dataset):
        """Locate the baseline metrics CSV, handling legacy filename variants."""
        p = directory / f'metrics_{dataset}.csv'
        if p.exists():
            return p
        p = directory / 'metrics.csv'
        if p.exists():
            return p
        candidates = sorted(f for f in directory.glob('metrics_*.csv')
                            if not f.stem.endswith('_val_mean'))
        return candidates[0] if candidates else None

    def _get(mname, src_type, detail, trim, dataset, is_synth, series_key):
        """Return (mse, mae) for one model / dataset / series combination."""
        if src_type == 'simulator':
            # Simod/AgentSimulator: results/<detail>/metrics_<detail>[_real].csv --
            # one combined file per simulator per log-group (real-life gained
            # Simod coverage 2026-08-28; AgentSimulator still synthetic-only,
            # so this naturally returns None/None for it on real-life datasets).
            key = (detail, is_synth)
            if key not in _sim_cache:
                suffix = '' if is_synth else '_real'
                p = RESULTS / detail / f'metrics_{detail}{suffix}.csv'
                _sim_cache[key] = pd.read_csv(p) if p.exists() else pd.DataFrame()
            sdf = _sim_cache[key]
            if sdf.empty:
                return None, None
            mask = (sdf['dataset'] == dataset) & (sdf['series'] == series_key)
            if not is_synth and 'trim' in sdf.columns:
                mask &= (sdf['trim'] == trim)
            row = sdf[mask]
            if row.empty:
                return None, None
            return float(row.iloc[0]['mse']), float(row.iloc[0]['mae'])

        if src_type == 'val_mean':
            if _vm_df.empty:
                return None, None
            mask = (_vm_df['dataset'] == dataset) & (_vm_df['series'] == series_key)
            rows = _vm_df[mask]
            if rows.empty:
                return None, None
            if 'trim' in rows.columns:
                t_rows = rows[rows['trim'] == trim]
                # No silent cross-trim fallback: a trim genuinely missing from
                # val_mean_all.csv (e.g. 'ssd', never computed there until
                # this was fixed) must show as missing, not silently borrow
                # another trim's number -- this is exactly what produced the
                # 'ssd' sheet's wrong val_average row (was quietly reading
                # trim='none' for every dataset).
                if t_rows.empty:
                    return None, None
                r = t_rows.iloc[0]
            else:
                r = rows.iloc[0]
            return float(r['mse']), float(r['mae'])

        elif src_type == 'baseline':
            key = ('synth', dataset) if is_synth else (trim, dataset)
            if key not in _bl_cache:
                if is_synth:
                    d = RESULTS / 'synthetic' / 'none' / dataset
                else:
                    d = RESULTS / trim / dataset
                p = _find_baseline_csv(d, dataset)
                _bl_cache[key] = pd.read_csv(p) if (p and p.exists()) else pd.DataFrame()
            df = _bl_cache[key]
            if df.empty:
                return None, None
            row = df[(df['model'] == detail) & (df['series'] == series_key)]
            if row.empty:
                return None, None
            return float(row.iloc[0]['mse']), float(row.iloc[0]['mae'])

        elif src_type == 'pmsd':
            # metrics_{dataset}_pmsd.csv lives right next to the main baseline
            # file, never merged into it — see MODELS comment above.
            key = ('pmsd', 'synth', dataset) if is_synth else ('pmsd', trim, dataset)
            if key not in _bl_cache:
                if is_synth:
                    d = RESULTS / 'synthetic' / 'none' / dataset
                else:
                    d = RESULTS / trim / dataset
                p = d / f'metrics_{dataset}_pmsd.csv'
                _bl_cache[key] = pd.read_csv(p) if p.exists() else pd.DataFrame()
            df = _bl_cache[key]
            if df.empty:
                return None, None
            row = df[(df['model'] == detail) & (df['series'] == series_key)]
            if row.empty:
                return None, None
            return float(row.iloc[0]['mse']), float(row.iloc[0]['mae'])

        elif src_type == 'foundation':
            # Real-life : results/reallife_ts/<trim>/<dataset>/
            # Synthetic : results/synthetic/none/<dataset>/  (same files as other baselines)
            key = ('reallife_ts', trim, dataset) if not is_synth else ('synth_fm', dataset)
            if key not in _bl_cache:
                if is_synth:
                    d = RESULTS / 'synthetic' / 'none' / dataset
                else:
                    d = RESULTS / 'reallife_ts' / trim / dataset
                p = _find_baseline_csv(d, dataset)
                _bl_cache[key] = pd.read_csv(p) if (p and p.exists()) else pd.DataFrame()
            df = _bl_cache[key]
            if df.empty:
                return None, None
            row = df[(df['model'] == detail) & (df['series'] == series_key)]
            if row.empty:
                return None, None
            return float(row.iloc[0]['mse']), float(row.iloc[0]['mae'])

        else:  # process
            subdir, suffix = detail[0], detail[1]
            fallback = detail[2] if len(detail) > 2 else None
            run = f'{dataset}_test_full'
            key = (subdir, trim, dataset, suffix)
            if key not in _pr_cache:
                p = RESULTS / subdir / trim / run / f'metrics_{run}{suffix}.csv'
                expected_model = subdir
                if not p.exists() and fallback:
                    p = RESULTS / fallback / trim / run / f'metrics_{run}{suffix}.csv'
                    expected_model = fallback
                if p.exists():
                    _df = pd.read_csv(p)
                    # Half-prefix process metrics files can carry TWO rows per
                    # series -- every camargo_hpo_half/metrics_*.csv (56/56,
                    # all real-life datasets x all trims -- confirmed
                    # universal) also has a duplicate 'camargo_hpo' row (the
                    # full regime's own value, seemingly copied in for
                    # reference). Filtering on series alone and taking
                    # iloc[0] silently picked whichever row comes first in
                    # the file -- always 'camargo_hpo', never the
                    # half-specific 'camargo_hpo_half' row -- so every
                    # camargo 'half' mode result in results_auto.xlsx was
                    # actually the full regime's number. subdir/fallback
                    # (e.g. 'camargo_hpo_half') IS the expected model label,
                    # so filter to it when present; bukhsh's own label is
                    # inconsistently 'bukhsh_hpo' vs 'bukhsh_hpo_half' across
                    # files with only ONE row each, so the filter is a no-op
                    # there either way.
                    if expected_model in set(_df['model']):
                        _df = _df[_df['model'] == expected_model]
                    _pr_cache[key] = _df
                else:
                    _pr_cache[key] = pd.DataFrame()
            df = _pr_cache[key]
            if df.empty:
                return None, None
            row = df[df['series'] == series_key]
            if row.empty:
                return None, None
            return float(row.iloc[0]['mse']), float(row.iloc[0]['mae'])

    # ── Style constants ──────────────────────────────────────────────────────
    _GREEN = PatternFill('solid', fgColor='C6EFCE')
    _BOLD  = Font(bold=True)
    _HDR1  = PatternFill('solid', fgColor='BDD7EE')   # blue  – series label row
    _HDR2  = PatternFill('solid', fgColor='D9E1F2')   # light – group / BV rows

    def _h(ws, row, col, val, fill=_HDR1):
        c = ws.cell(row=row, column=col, value=val)
        c.font = _BOLD
        c.fill = fill
        c.alignment = Alignment(horizontal='center')
        return c

    def _n(ws, row, col, val):
        if val is None or (isinstance(val, float) and np.isnan(val)):
            return ws.cell(row=row, column=col)
        c = ws.cell(row=row, column=col, value=round(val, 4))
        c.number_format = '0.0000'
        return c

    # ── Column layout (1-indexed, 3 cols per dataset: mse, rmse, mae) ────────
    # 'none' sheet:
    #   col  1     : model name
    #   col  2-16  : Loan  (5 datasets × 3 cols each)
    #   col  17    : [gap]
    #   col  18-32 : O2C   (5 datasets × 3 cols each)
    #   col  33    : [gap]
    #   col  34-36 : Best Value synthetic (mse, rmse, mae)
    #   col  37-39 : [gap]
    #   col  40-63 : Real-Life (8 datasets × 3 cols each)
    #   col  64    : [gap]
    #   col  65-67 : Best Value real-life (mse, rmse, mae)
    #
    # other sheets:
    #   col  1     : model name
    #   col  2-25  : Real-Life (8 datasets × 3 cols each)
    #   col  26    : [gap]
    #   col  27-29 : Best Value real-life (mse, rmse, mae)

    def _write_block(ws, r0, trim, is_none, series_label, series_key):
        """Write one CC or TT block. Returns the next free row."""
        r = r0

        if is_none:
            loan_triples = [(2  + i*3, 3  + i*3, 4  + i*3) for i in range(5)]
            o2c_triples  = [(18 + i*3, 19 + i*3, 20 + i*3) for i in range(5)]
            real_triples = [(40 + i*3, 41 + i*3, 42 + i*3) for i in range(8)]
            synth_triples = loan_triples + o2c_triples
            bv_synth     = (34, 35, 36)   # mse, rmse, mae
            bv_real      = (65, 66, 67)
        else:
            loan_triples  = []
            o2c_triples   = []
            real_triples  = [(2 + i*3, 3 + i*3, 4 + i*3) for i in range(8)]
            synth_triples = []
            bv_synth      = None
            bv_real       = (27, 28, 29)

        all_triples = synth_triples + real_triples

        # Row A – series label
        _h(ws, r, 2, series_label)
        r += 1

        # Row B – group labels (none sheet only)
        if is_none:
            _h(ws, r, 2,  'Loan (simple)', fill=_HDR2)
            _h(ws, r, 18, 'O2C (complex)', fill=_HDR2)
            _h(ws, r, 40, 'Real Life',     fill=_HDR2)
        r += 1

        # Row C – dataset names + Best Value labels
        if is_none:
            for i, sn in enumerate(SHORT_NAMES):
                ws.cell(row=r, column=2  + i*3, value=sn).font = _BOLD
                ws.cell(row=r, column=18 + i*3, value=sn).font = _BOLD
            for i, ds in enumerate(REAL_LIFE):
                ws.cell(row=r, column=40 + i*3, value=ds).font = _BOLD
            _h(ws, r, bv_synth[0], 'Best Value', fill=_HDR2)
            _h(ws, r, bv_real[0],  'Best Value', fill=_HDR2)
        else:
            for i, ds in enumerate(REAL_LIFE):
                ws.cell(row=r, column=2 + i*3, value=ds).font = _BOLD
            _h(ws, r, bv_real[0], 'Best Value', fill=_HDR2)
        r += 1

        # Row D – column sub-headers  model | mse rmse mae | mse rmse mae | …
        ws.cell(row=r, column=1, value='model').font = _BOLD
        for cm, cr, ca in all_triples:
            ws.cell(row=r, column=cm, value='mse').font  = _BOLD
            ws.cell(row=r, column=cr, value='rmse').font = _BOLD
            ws.cell(row=r, column=ca, value='mae').font  = _BOLD
        ws.cell(row=r, column=bv_real[0], value='mse').font  = _BOLD
        ws.cell(row=r, column=bv_real[1], value='rmse').font = _BOLD
        ws.cell(row=r, column=bv_real[2], value='mae').font  = _BOLD
        if is_none and bv_synth:
            ws.cell(row=r, column=bv_synth[0], value='mse').font  = _BOLD
            ws.cell(row=r, column=bv_synth[1], value='rmse').font = _BOLD
            ws.cell(row=r, column=bv_synth[2], value='mae').font  = _BOLD
        r += 1

        # ── Data rows ────────────────────────────────────────────────────────
        data_r0     = r
        col_entries = {}   # col → [(row_idx, value, cell)]

        for mi, (mname, src_type, detail) in enumerate(MODELS):
            ws.cell(row=r, column=1, value=mname)

            if is_none:
                for i, ds in enumerate(SYNTH_LOAN):
                    mse, mae = _get(mname, src_type, detail, 'none', ds, True, series_key)
                    rmse = np.sqrt(mse) if mse is not None else None
                    cm, cr, ca = loan_triples[i]
                    col_entries.setdefault(cm, []).append((r, mse,  _n(ws, r, cm, mse)))
                    col_entries.setdefault(cr, []).append((r, rmse, _n(ws, r, cr, rmse)))
                    col_entries.setdefault(ca, []).append((r, mae,  _n(ws, r, ca, mae)))
                for i, ds in enumerate(SYNTH_O2C):
                    mse, mae = _get(mname, src_type, detail, 'none', ds, True, series_key)
                    rmse = np.sqrt(mse) if mse is not None else None
                    cm, cr, ca = o2c_triples[i]
                    col_entries.setdefault(cm, []).append((r, mse,  _n(ws, r, cm, mse)))
                    col_entries.setdefault(cr, []).append((r, rmse, _n(ws, r, cr, rmse)))
                    col_entries.setdefault(ca, []).append((r, mae,  _n(ws, r, ca, mae)))

            for i, ds in enumerate(REAL_LIFE):
                mse, mae = _get(mname, src_type, detail, trim, ds, False, series_key)
                rmse = np.sqrt(mse) if mse is not None else None
                cm, cr, ca = real_triples[i]
                col_entries.setdefault(cm, []).append((r, mse,  _n(ws, r, cm, mse)))
                col_entries.setdefault(cr, []).append((r, rmse, _n(ws, r, cr, rmse)))
                col_entries.setdefault(ca, []).append((r, mae,  _n(ws, r, ca, mae)))

            r += 1

        # ── Highlight column minimums + count Best Value wins ────────────────
        # col_meta: col → (in_synth: bool, metric_type: str)
        col_meta = {}
        if is_none:
            for cm, cr, ca in synth_triples:
                col_meta[cm] = (True,  'mse')
                col_meta[cr] = (True,  'rmse')
                col_meta[ca] = (True,  'mae')
        for cm, cr, ca in real_triples:
            col_meta[cm] = (False, 'mse')
            col_meta[cr] = (False, 'rmse')
            col_meta[ca] = (False, 'mae')

        synth_bv_mse  = [0] * N_MODELS
        synth_bv_rmse = [0] * N_MODELS
        synth_bv_mae  = [0] * N_MODELS
        real_bv_mse   = [0] * N_MODELS
        real_bv_rmse  = [0] * N_MODELS
        real_bv_mae   = [0] * N_MODELS

        for col, entries in col_entries.items():
            valid = [(ri, v, c) for ri, v, c in entries
                     if v is not None and not (isinstance(v, float) and np.isnan(v))]
            if len(valid) < 2:
                continue
            min_v = min(v for _, v, _ in valid)
            in_synth, metric_type = col_meta.get(col, (False, 'mse'))
            for ri, v, c in valid:
                if abs(v - min_v) < 1e-9:
                    c.fill = _GREEN
                    mi = ri - data_r0
                    if in_synth:
                        if metric_type == 'mse':   synth_bv_mse[mi]  += 1
                        elif metric_type == 'rmse': synth_bv_rmse[mi] += 1
                        else:                       synth_bv_mae[mi]  += 1
                    else:
                        if metric_type == 'mse':   real_bv_mse[mi]   += 1
                        elif metric_type == 'rmse': real_bv_rmse[mi]  += 1
                        else:                       real_bv_mae[mi]   += 1

        # Write Best Value counts
        for mi in range(N_MODELS):
            dr = data_r0 + mi
            ws.cell(row=dr, column=bv_real[0], value=real_bv_mse[mi]  or None)
            ws.cell(row=dr, column=bv_real[1], value=real_bv_rmse[mi] or None)
            ws.cell(row=dr, column=bv_real[2], value=real_bv_mae[mi]  or None)
            if is_none and bv_synth:
                ws.cell(row=dr, column=bv_synth[0], value=synth_bv_mse[mi]  or None)
                ws.cell(row=dr, column=bv_synth[1], value=synth_bv_rmse[mi] or None)
                ws.cell(row=dr, column=bv_synth[2], value=synth_bv_mae[mi]  or None)

        return r + 2   # 2 blank rows between blocks

    # ── Build workbook ───────────────────────────────────────────────────────
    wb = Workbook()
    wb.remove(wb.active)

    for trim in TRIM_SHEETS:
        ws      = wb.create_sheet(title=trim)
        is_none = (trim == 'none')
        row = 1
        for series_label, series_key in SERIES_LIST:
            row = _write_block(ws, row, trim, is_none, series_label, series_key)

        ws.column_dimensions['A'].width = 26
        for col in range(2, 75):
            ws.column_dimensions[get_column_letter(col)].width = 10

    wb.save(str(out_path))
    print(f'Saved → {out_path}')
    return out_path


export_results_excel()

## Real-life (ssd) — best-of-category `_grid.pdf` charts

In [ ]:
import sys
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(ROOT / 'analysis'))
from split_correlation import (
    NAIVE_BASELINES, STATISTICAL, FOUNDATION_MODELS, SIMULATION_MODELS, PROCESS_MODELS,
    PPM_REGIMES, DISPLAY_NAMES as _SC_DISPLAY_NAMES, SERIES,
    parse_approach_maes, parse_ppm_maes, real_logs, REAL_TRIM_NAMES,
)

AGG_OUT_DIR = ROOT / 'results' / 'aggregated_method_comparison'
AGG_OUT_DIR.mkdir(parents=True, exist_ok=True)

LOAN_LOGS = [p.stem for p in sorted((ROOT / 'data' / 'synthetic').glob('loan_*.xes'))
            if 'recency' not in p.stem]
O2C_LOGS  = [p.stem for p in sorted((ROOT / 'data' / 'synthetic').glob('o2c_*.xes'))
            if 'recency' not in p.stem]

# ML group uses N-BEATS, not split_correlation.py's N-HiTS (that module's paper set
# uses nhits — this notebook's method list intentionally differs, per explicit request).
# parse_approach_maes() doesn't fetch 'nbeats' (it's outside its own hardcoded model
# list), so it's read directly from the same metrics_<dataset>.csv here instead.
DISPLAY_NAMES = {**_SC_DISPLAY_NAMES, 'nbeats': 'N-BEATS'}
AGG_ML_MODELS = ['ridge', 'ridge_mimo', 'nbeats', 'tft']

def _nbeats_mae(dataset, trim, is_real):
    d = (ROOT / 'results' / trim / dataset) if is_real else (ROOT / 'results' / 'synthetic' / 'none' / dataset)
    p = d / f'metrics_{dataset}.csv'
    if not p.exists():
        return {}
    df = pd.read_csv(p)
    sub = df[df['model'] == 'nbeats']
    return dict(zip(sub['series'], sub['mae'])) if not sub.empty else {}

PPM_ABBR = {'plain_field': 'PF', 'half': 'Half', 'full': 'Full'}
METHOD_ORDER = NAIVE_BASELINES + STATISTICAL + AGG_ML_MODELS + FOUNDATION_MODELS + SIMULATION_MODELS

def _label(model):
    return DISPLAY_NAMES.get(model, model)

# Fixed x-axis order across every chart, so bars land in the same position
# regardless of which methods happen to have data for a given group/trim.
ALL_LABELS = (
    [_label(m) for m in METHOD_ORDER]
    + [f'{_label(m)} ({PPM_ABBR[r]})' for m in PROCESS_MODELS for r in PPM_REGIMES]
)

LABEL_FAMILY = {}
for m in NAIVE_BASELINES: LABEL_FAMILY[_label(m)] = 'baseline'
for m in STATISTICAL:     LABEL_FAMILY[_label(m)] = 'statistical'
for m in AGG_ML_MODELS:   LABEL_FAMILY[_label(m)] = 'ml'
LABEL_FAMILY[_label('chronos')] = 'chronos'
LABEL_FAMILY[_label('tabpfn')]  = 'tabpfn'
# 'simulation' covers all of SIMULATION_MODELS (currently just PMSD, with two more
# System Dynamics-style simulation baselines to be added later) -- looping over the
# list (like every other family below) instead of a pmsd-only one-off means new
# entries added to SIMULATION_MODELS automatically join the same visual group
# (shared color) without needing another edit here.
for m in SIMULATION_MODELS: LABEL_FAMILY[_label(m)] = 'simulation'
for m in PROCESS_MODELS:
    for r in PPM_REGIMES:
        LABEL_FAMILY[f'{_label(m)} ({PPM_ABBR[r]})'] = 'ppm'

# Categorical hues — same assignments as analysis/ts_comparison.py's plot_comparison
# (COLOR_NAIVE/COLOR_STAT/COLOR_ML/COLOR_CHRONOS/COLOR_TABPFN/COLOR_PPM), kept in
# sync manually since this notebook doesn't import that module's plotting code.
FAMILY_COLOR = {
    'baseline': '#8a8a86', 'statistical': '#2a78d6', 'ml': '#eb6834',
    'chronos': '#4a3aa7', 'tabpfn': '#008300', 'simulation': '#eda100', 'ppm': '#e34948',
}
# PPM regime -> alpha: plain-field (least prior knowledge) lightest, full-trace
# (most prior knowledge) darkest — a sequential ramp within the one PPM hue.
PPM_REGIME_ALPHA = {'PF': 0.45, 'Half': 0.7, 'Full': 1.0}

def _bar_color_alpha(label):
    fam = LABEL_FAMILY.get(label, 'baseline')
    color = FAMILY_COLOR.get(fam, '#999999')
    alpha = 1.0
    if fam == 'ppm':
        for tag, a in PPM_REGIME_ALPHA.items():
            if label.endswith(f'({tag})'):
                alpha = a
                break
    return color, alpha


def average_mae_by_method(datasets, trim, is_real):
    """Returns {method_label: {series: avg_mae_or_None}} averaged across `datasets`
    for one trim. PPM labels carry a regime suffix, e.g. 'GLSTM (Full)'. A method
    missing for some datasets is averaged only over the datasets where it's present
    (not zero-filled) — coverage gaps show as a shorter bar's worth of data, not a
    misleadingly low average."""
    sums = defaultdict(lambda: {s: [] for s in SERIES})
    for ds in datasets:
        non_ppm = parse_approach_maes(ds, trim, is_real)
        non_ppm['nbeats'] = _nbeats_mae(ds, trim, is_real)
        for model in METHOD_ORDER:
            if model in non_ppm:
                for s in SERIES:
                    v = non_ppm[model].get(s)
                    if v is not None:
                        sums[_label(model)][s].append(v)
        for regime in PPM_REGIMES:
            ppm = parse_ppm_maes(ds, trim, regime)
            for model in PROCESS_MODELS:
                if model in ppm:
                    label = f'{_label(model)} ({PPM_ABBR[regime]})'
                    for s in SERIES:
                        v = ppm[model].get(s)
                        if v is not None:
                            sums[label][s].append(v)
    out = {}
    for label, d in sums.items():
        out[label] = {s: (float(np.mean(d[s])) if d[s] else None) for s in SERIES}
        out[label]['n'] = {s: len(d[s]) for s in SERIES}
    return out


def plot_method_bars(data_by_group, series_key, series_label, title, save_path=None, ylabel=None):
    """data_by_group: {group_label: average_mae_by_method(...) result}. Pass a
    single {'': agg} for an ungrouped (real-life, one trim) chart, or two+ entries
    (e.g. {'Loan': .., 'O2C': ..}) for a side-by-side grouped chart.

    ylabel: overrides the default f'{series_label} MAE' y-axis label -- needed
    for non-MAE values (e.g. NMAE %) passed in via data_by_group, where the
    hardcoded ' MAE' suffix would be wrong. Leaving it None keeps every
    existing call site's behavior unchanged."""
    labels = [l for l in ALL_LABELS
             if any(l in g and g[l].get(series_key) is not None for g in data_by_group.values())]
    groups = list(data_by_group.keys())
    n_groups = len(groups)
    x = np.arange(len(labels))
    width = 0.8 / max(n_groups, 1)

    fig, ax = plt.subplots(figsize=(max(12, len(labels) * 0.55), 5))
    for gi, g in enumerate(groups):
        offset = (gi - (n_groups - 1) / 2) * width
        for xi, l in zip(x, labels):
            v = data_by_group[g].get(l, {}).get(series_key)
            if v is None:
                continue
            color, alpha = _bar_color_alpha(l)
            ax.bar(xi + offset, v, width, color=color, alpha=alpha,
                  edgecolor='white', linewidth=0.5,
                  hatch=('' if gi == 0 else '//') if n_groups > 1 else None)
        if n_groups > 1:
            ax.bar(np.nan, 0, width, color='#999999',
                  hatch=('' if gi == 0 else '//'), label=g, edgecolor='white')

    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=60, ha='right', fontsize=8)
    ax.set_ylabel(ylabel if ylabel is not None else f'{series_label} MAE')
    ax.set_ylim(bottom=0)
    ax.set_title(title)
    ax.grid(axis='y', alpha=0.25)
    if n_groups > 1:
        ax.legend(loc='upper right', frameon=False)
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, bbox_inches='tight')
    plt.show()
    return fig

SERIES_LABELS = {'concurrent_cases': 'Concurrent cases', 'throughput_time': 'Avg. throughput time (h)'}

In [ ]:
# 'Simulation' sits between Tab. FM and the three PPM categories -- same relative
# order as SIMULATION_MODELS in METHOD_ORDER (split_correlation.py), which is placed
# after FOUNDATION_MODELS and before PROCESS_MODELS for the same reason.
CATEGORY_LABELS = ['Baselines', 'Stat. TS', 'ML TS', 'Tab. FM', 'Simulation', 'Plain-field', 'Full PPM', 'Half PPM']
CATEGORY_COLORS = dict(zip(CATEGORY_LABELS,
    ['#C0C0C0', '#BDD7EE', '#6FA8DC', '#C9B1D9', '#eda100', '#F4B183', '#A9D18E', '#70AD47']))

def _label_category(label):
    fam = LABEL_FAMILY.get(label, 'baseline')
    if fam == 'baseline':    return 'Baselines'
    if fam == 'statistical': return 'Stat. TS'
    if fam == 'ml':          return 'ML TS'
    if fam in ('chronos', 'tabpfn'): return 'Tab. FM'
    if fam == 'simulation':  return 'Simulation'
    if fam == 'ppm':
        if label.endswith('(PF)'):   return 'Plain-field'
        if label.endswith('(Full)'): return 'Full PPM'
        if label.endswith('(Half)'): return 'Half PPM'
    return None

def best_of_category(agg):
    """Collapse a per-method average_mae_by_method() result down to one value per
    method category, taking the best (lowest) MAE within each category."""
    out = {cat: {s: None for s in SERIES} for cat in CATEGORY_LABELS}
    for label, vals in agg.items():
        cat = _label_category(label)
        if cat is None:
            continue
        for s in SERIES:
            v = vals.get(s)
            if v is None:
                continue
            cur = out[cat][s]
            out[cat][s] = v if cur is None else min(cur, v)
    return out

# Optional styling knobs (all default to the original look, so the synthetic
# Loan/O2C calls at the bottom of this cell are unaffected) -- added for the
# real-life/ssd bestcat call in the next cell:
#   show_title        : drop the header text entirely
#   bar_width         : < 0.8 (matplotlib default-ish) for slimmer bars
#   show_value_labels : print each bar's MAE above it
#   group_ppm         : Plain-field/Full PPM/Half PPM share one hue (the same
#                       red used for the PPM family in plot_method_bars above)
#                       at different alpha (PPM_REGIME_ALPHA), instead of three
#                       unrelated colors -- visually ties them together as
#                       regime-variants of the same underlying PPM concept.
#                       Instead of a legend box, a bracket + label is drawn
#                       directly below the bars/tick labels (mirrors
#                       error_direction.ipynb's plot_bias_grouped2 grouping
#                       style, just placed under the tick labels rather than
#                       above the axes) -- a legend box used to collide with
#                       whichever bar happened to be tallest near that corner.
#   fontsize          : base font size for ticks/labels/legend (bold throughout)


In [ ]:
trim = 'ssd'

SSD_CATEGORY_LABELS = ['Baselines', 'Stat. TS', 'ML TS', 'Tab. FM', 'Simulation', 'Plain', 'First', 'Half']
SSD_PPM_ALPHA = {'Plain': PPM_REGIME_ALPHA['PF'], 'First': PPM_REGIME_ALPHA['Full'], 'Half': PPM_REGIME_ALPHA['Half']}


In [ ]:
_CAT_RENAME = {"Plain-field": "Plain", "Full PPM": "First", "Half PPM": "Half"}


def per_dataset_category_best(datasets, trim, is_real, series_key):
    """{dataset: {ssd_category: best_mae}} -- best_of_category() per dataset,
    SSD-renamed (Plain/First/Half)."""
    out = {}
    for ds in datasets:
        agg_one = average_mae_by_method([ds], trim, is_real)
        best_one = best_of_category(agg_one)
        out[ds] = {}
        for cat in CATEGORY_LABELS:
            v = best_one[cat][series_key]
            if v is not None:
                out[ds][_CAT_RENAME.get(cat, cat)] = v
    return out


In [ ]:
def plot_category_bars_grid(datasets, trim, is_real, series_key, series_label,
                            category_labels, ppm_alpha, save_path=None, fontsize=24,
                            show_legend=True):
    """Paper-ready (ACM format) version of your original ask: one best-of-category
    bar chart per dataset, 8 across, no cross-dataset aggregation. No text sits
    above any panel (no suptitle, no per-panel title) -- each panel's dataset
    name moves BELOW its numbered x-axis instead. The long, rotated category
    names are replaced by a single number->category legend along the TOP of the
    whole figure, using circled-digit glyphs (matching the bars' own numbered
    ticks) instead of a plain "N." prefix. Each panel keeps its own y-axis
    scale (not shared). Uses seaborn's "talk"/whitegrid theme for larger,
    unbolded, print-friendly typography -- sized for the figure's REAL final
    width in the paper, not the wide canvas this renders at in the notebook,
    so avoid growing figsize further to "fix" crowding; shrink fontsize or the
    number of panels per row instead."""
    import matplotlib as mpl
    import seaborn as sns
    mpl.rcParams["pdf.fonttype"] = 42   # embed fonts as Type-1 (LaTeX-friendly,
    mpl.rcParams["ps.fonttype"] = 42    # matches export_viz_pdf's own convention)
    sns.set_theme(context="talk", style="whitegrid")
    plt.rcParams["font.family"] = "DejaVu Sans"  # seaborn's "talk" theme switches to
    # Arial, which lacks the circled-digit glyphs (U+2460..) used below -- they
    # silently vanish (a UserWarning, not an error) unless forced back to a font
    # that covers this Unicode range.

    per_ds = per_dataset_category_best(datasets, trim, is_real, series_key)
    all_vals = [v for cats in per_ds.values() for v in cats.values()]
    # 4+ digit y-tick labels (>=1000) are wide enough to crowd the neighboring
    # panel at the normal gap -- widen it a bit whenever any panel needs one.
    wspace = 0.5 if (all_vals and max(all_vals) >= 1000) else 0.35
    n = len(datasets)
    fig, axes = plt.subplots(1, n, figsize=(n * 3.2, 5.6))
    axes = np.atleast_1d(axes)

    # Fixed numbering across ALL panels (not just whichever categories a given
    # panel happens to have) so a given circled digit means the same category
    # in every panel. CIRCLED_DIGIT reuses the same glyph family Unicode ships
    # for exactly this ("circled number") use, U+2460.. -- no custom marker
    # drawing needed, and it survives Type-42 PDF embedding (DejaVu Sans covers
    # this range).
    all_cats = [c for c in category_labels if any(c in per_ds.get(ds, {}) for ds in datasets)]
    cat_number = {c: i + 1 for i, c in enumerate(all_cats)}
    circled = lambda i: chr(0x2460 + i - 1)

    for ax, ds in zip(axes, datasets):
        cats = [c for c in category_labels if c in per_ds.get(ds, {})]
        vals = [per_ds[ds][c] for c in cats]
        colors = [FAMILY_COLOR["ppm"] if c in ppm_alpha else CATEGORY_COLORS.get(c, "#999999") for c in cats]
        alphas = [ppm_alpha.get(c, 1.0) for c in cats]
        for i, (v, col, a) in enumerate(zip(vals, colors, alphas)):
            ax.bar(i, v, width=0.72, color=col, alpha=a, edgecolor="white", linewidth=0.6)
        ax.set_xticks(range(len(cats)))
        ax.set_xticklabels([])
        ax.tick_params(axis="x", length=0)
        # Two staggered ("versetzt") rows instead of one crowded row -- lets the
        # digits run much larger without adjacent circles touching. Both rows and
        # the dataset name below use the SAME axes-fraction transform so their
        # vertical order is guaranteed regardless of fontsize (labelpad, in points,
        # can't be reasoned about against an axes-fraction row position).
        row_y = {0: -0.07, 1: -0.20}
        for i, c in enumerate(cats):
            ax.text(i, row_y[i % 2], circled(cat_number[c]), transform=ax.get_xaxis_transform(),
                    ha="center", va="top", fontsize=fontsize + 10, clip_on=False)
        ax.set_ylim(bottom=0)
        ds_short = ds.replace("bpic", "bpi").replace("-dom", "-d").replace("-int", "-i")
        ax.text(0.5, -0.34, ds_short, transform=ax.transAxes, ha="center", va="top", fontsize=fontsize)
        ax.xaxis.grid(False)
        ax.tick_params(axis="y", labelsize=fontsize - 2)
        ax.yaxis.set_major_locator(plt.MaxNLocator(nbins=4))
        for spine in ("top", "right"):
            ax.spines[spine].set_visible(False)

    if show_legend:
        from matplotlib.colors import to_rgba
        import matplotlib.patches as mpatches
        import matplotlib.text as mtext
        from matplotlib.legend_handler import HandlerBase

        class _CircleDigitHandler(HandlerBase):
            """Plain BLACK digit glyph (matching the x-axis exactly, no fill --
            the axis circles are just text, not colored markers) plus a
            SEPARATE small colored swatch next to it for the category color.
            Digit sized independently of the legend's own (normal-sized) text
            fontsize so it can match the x-axis glyph size without inflating
            "Baselines"/"Simulation" past the dataset-name size, and keeps the
            whole legend narrow enough to fit in one row."""
            def __init__(self, digit, color, alpha, digit_fontsize):
                self.digit, self.color, self.alpha = digit, color, alpha
                self.digit_fontsize = digit_fontsize
                super().__init__()

            def create_artists(self, legend, orig_handle, xdescent, ydescent,
                               width, height, fontsize, trans):
                cy = height / 2 - ydescent
                sw_w, sw_h = width * 0.42, height * 0.85
                sw_x = 0.0 - xdescent
                swatch = mpatches.Rectangle((sw_x, cy - sw_h / 2), sw_w, sw_h,
                                            facecolor=to_rgba(self.color, self.alpha),
                                            edgecolor="white", linewidth=0.8, transform=trans)
                digit_x = width * 0.75 - xdescent
                txt = mtext.Text(digit_x, cy, self.digit, fontsize=self.digit_fontsize,
                                 color="black", ha="center", va="center", transform=trans)
                return [swatch, txt]

        legend_handles, handler_map = [], {}
        for c in all_cats:
            color = FAMILY_COLOR["ppm"] if c in ppm_alpha else CATEGORY_COLORS.get(c, "#999999")
            alpha = ppm_alpha.get(c, 1.0)
            h = mpatches.Patch(alpha=0)  # invisible placeholder handle
            handler_map[h] = _CircleDigitHandler(circled(cat_number[c]), color, alpha, fontsize + 8)
            legend_handles.append((h, c))

        fig.legend(handles=[h for h, _ in legend_handles], labels=[c for _, c in legend_handles],
                  handler_map=handler_map, loc="lower center", ncol=len(all_cats),
                  fontsize=fontsize, frameon=False, bbox_to_anchor=(0.5, 1.02),
                  handletextpad=0.5, columnspacing=1.3, handlelength=2.4, handleheight=1.8)

    fig.canvas.draw()
    fig.subplots_adjust(left=0.05, right=0.995, bottom=0.30, top=0.95, wspace=wspace)
    if save_path:
        fig.savefig(save_path, bbox_inches="tight")
    plt.show()
    return fig


# CTP/TTP -- the paper's own short axis-label abbreviations, used only for this
# grid chart's y-axis label (not a change to the shared SERIES_LABELS dict, which
# every other chart in this notebook still reads for its own longer labels).
GRID_SERIES_LABELS = {"concurrent_cases": "CTP", "throughput_time": "TTP"}

for series_key, series_label in GRID_SERIES_LABELS.items():
    suffix = "cc" if series_key == "concurrent_cases" else "tt"
    plot_category_bars_grid(
        real_logs, trim, True, series_key, series_label,
        category_labels=SSD_CATEGORY_LABELS, ppm_alpha=SSD_PPM_ALPHA,
        save_path=AGG_OUT_DIR / f"real_{trim}_bestcat_{suffix}_grid.pdf",
        show_legend=(series_key == "concurrent_cases"),  # only the top (CC) chart needs it
    )